# RFM Segmentation & Retention Strategy
## Part 2 - Customer Segmentation for Retention

**Objective:** Build RFM-based customer segments and recommend targeted retention actions.

**Dataset:** 2,400 customers with order, support, and behavioral data (snapshot: 2025-09-30)

**Segments Created:** 9 customer segments

**Key Finding:** 27% of customers are At-Risk and need immediate retention attention.

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('c:/Users/Asus/Downloads/rfm_modeling_snapshot (1).csv')
print(f"Dataset: {len(df)} customers, {df.shape[1]} features")
df.head()

## 2. RFM Feature Creation

**Recency (R):** Days since last purchase - lower is better
**Frequency (F):** Number of purchases in last 180 days - higher is better
**Monetary (M):** Total spend in last 180 days - higher is better

Each score is 1-5 (quintiles), where 5 = best behavior.

In [ ]:
# Create RFM scores
rfm = df[['customer_id', 'recency_days', 'frequency_180d', 'monetary_180d',
          'return_rate_180d', 'avg_discount_pct_180d', 'avg_rating_180d',
          'ticket_count_90d', 'negative_ticket_rate_90d',
          'sessions_30d', 'campaign_clicks_30d', 'email_opens_30d',
          'churn_next_60d', 'split', 'loyalty_tier']].copy()

# R Score: lower recency = higher score
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# F Score: higher frequency = higher score
rfm['F_score'] = pd.qcut(rfm['frequency_180d'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

# M Score: higher monetary = higher score
rfm['M_score'] = pd.qcut(rfm['monetary_180d'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Handle zeros
rfm.loc[rfm['frequency_180d'] == 0, 'F_score'] = 1
rfm.loc[rfm['monetary_180d'] == 0, 'M_score'] = 1

# Combined score
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print("RFM Statistics:")
print(rfm[['recency_days', 'frequency_180d', 'monetary_180d', 'RFM_score']].describe())
print(f"\nRFM Score Range: {rfm['RFM_score'].min()} to {rfm['RFM_score'].max()}")

## 3. Segmentation Logic

**9 Customer Segments:**

| Segment | Criteria | Description |
|---------|----------|-------------|
| **Champions** | R≥4, F≥4, M≥4 | Best customers - recent, frequent, high spenders |
| **Loyal Customers** | F≥4, R≥3, M≥3 | Frequent buyers with consistent relationship |
| **New Customers** | R≥4, F≤2, F>0 | Recent first-time buyers |
| **At-Risk Customers** | R≤2, F>0 | Previously active but not recent (27% of base) |
| **Dormant Customers** | R>180, F>0 | Inactive 6+ months but had history |
| **Lost Customers** | F=0, R>200 | No purchases, very high recency |
| **High-Value but Unhappy** | M≥4 + (rating≤2.5 OR return≥25% OR neg_ticket≥50%) | High spenders with service issues |
| **Discount-Sensitive** | F≥3, discount≥35%, M≤2 | Frequent but low-value, discount-dependent |
| **Potential Loyalists** | R≥3, F≥2, M≥2 | Moderate engagement, growth potential |
| **Hibernating** | R≤2, F≤2, M≤2, F>0 | Low activity, not completely lost |
| **Other** | Default | Mixed patterns |

In [ ]:
def assign_segment(row):
    R, F, M = row['R_score'], row['F_score'], row['M_score']
    rfn, fqn = row['recency_days'], row['frequency_180d']
    ret = row['return_rate_180d']
    neg_ticket = row['negative_ticket_rate_90d']
    rating = row['avg_rating_180d']
    
    # Champions
    if R >= 4 and F >= 4 and M >= 4:
        return 'Champions'
    # Loyal Customers
    if F >= 4 and R >= 3 and M >= 3:
        return 'Loyal Customers'
    # New Customers
    if R >= 4 and F <= 2 and fqn > 0:
        return 'New Customers'
    # Dormant Customers
    if rfn > 180 and fqn > 0:
        return 'Dormant Customers'
    # At-Risk Customers
    if R <= 2 and fqn > 0:
        return 'At-Risk Customers'
    # High-Value but Unhappy
    if M >= 4 and (rating <= 2.5 or ret >= 0.25 or neg_ticket >= 0.5):
        return 'High-Value but Unhappy'
    # Discount-Sensitive
    if F >= 3 and row['avg_discount_pct_180d'] >= 0.35 and M <= 2:
        return 'Discount-Sensitive Customers'
    # Potential Loyalists
    if R >= 3 and F >= 2 and M >= 2:
        return 'Potential Loyalists'
    # Hibernating
    if R <= 2 and F <= 2 and M <= 2 and fqn > 0:
        return 'Hibernating Customers'
    # Lost Customers
    if fqn == 0 and rfn > 200:
        return 'Lost Customers'
    # Default
    return 'Other'

rfm['segment_name'] = rfm.apply(assign_segment, axis=1)

print("\nSegment Distribution:")
print(rfm['segment_name'].value_counts())
print(f"\nTotal Segments: {rfm['segment_name'].nunique()}")

## 4. Additional Behavioral & Support Signals

**Three key non-RFM signals added:**
1. **Support Complaint Score** - combines ticket count and negative ticket rate
2. **Engagement Score** - combines web/app sessions, product views, cart adds, email opens, campaign clicks
3. **Discount Dependency** - average discount percentage used

In [ ]:
# Support complaint score (0-1 scale)
rfm['support_complaint_score'] = (
    0.5 * (rfm['ticket_count_90d'] / rfm['ticket_count_90d'].max()) +
    0.5 * rfm['negative_ticket_rate_90d']
)

# Engagement score (0-1 scale)
rfm['engagement_score'] = (
    0.3 * (rfm['sessions_30d'] / rfm['sessions_30d'].max()) +
    0.2 * (rfm['campaign_clicks_30d'] / rfm['campaign_clicks_30d'].max()) +
    0.2 * (rfm['email_opens_30d'] / rfm['email_opens_30d'].max()) +
    0.3 * (rfm['sessions_30d'] / rfm['sessions_30d'].max())
)

# Discount dependency
rfm['discount_dependency'] = rfm['avg_discount_pct_180d']

print("Additional signals created successfully")
print("\nSample of signals by segment:")
rfm.groupby('segment_name')[['support_complaint_score', 'engagement_score', 'discount_dependency']].mean().round(3)

## 5. Segment Summary & Insights

In [ ]:
# Segment summary
summary = rfm.groupby('segment_name').agg({
    'customer_id': 'count',
    'recency_days': 'mean',
    'frequency_180d': 'mean',
    'monetary_180d': 'mean',
    'churn_next_60d': 'mean',
    'support_complaint_score': 'mean',
    'engagement_score': 'mean',
    'discount_dependency': 'mean'
}).round(3)

summary.columns = ['Count', 'Avg Recency', 'Avg Frequency', 'Avg Monetary', 
                   'Churn Rate', 'Support Score', 'Engagement', 'Discount Dep']
summary['Pct'] = (summary['Count'] / len(rfm) * 100).round(1)
summary = summary.sort_values('Count', ascending=False)
summary

## 6. Visualizations

In [ ]:
# Chart 1: Segment Distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('RFM Segmentation Analysis', fontsize=14, fontweight='bold')

# Segment counts
ax1 = axes[0, 0]
counts = rfm['segment_name'].value_counts()
ax1.barh(counts.index, counts.values, color=plt.cm.Set3(np.linspace(0, 1, len(counts))))
ax1.set_xlabel('Number of Customers')
ax1.set_title('Segment Distribution')
ax1.grid(axis='x', alpha=0.3)

# Churn rate by segment
ax2 = axes[0, 1]
churn = rfm.groupby('segment_name')['churn_next_60d'].mean().reindex(counts.index)
colors = ['green' if x < 0.3 else 'orange' if x < 0.6 else 'red' for x in churn.values]
ax2.barh(churn.index, churn.values, color=colors)
ax2.set_xlabel('Churn Rate')
ax2.set_title('Churn Risk by Segment')
ax2.axvline(x=rfm['churn_next_60d'].mean(), color='blue', linestyle='--', label='Avg')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

# RFM heatmap
ax3 = axes[1, 0]
rfm_heatmap = rfm.groupby('segment_name')[['R_score', 'F_score', 'M_score']].mean()
rfm_heatmap = rfm_heatmap.reindex(counts.index)
sns.heatmap(rfm_heatmap, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax3, vmin=1, vmax=5)
ax3.set_title('Average RFM Scores')

# Monetary value by segment
ax4 = axes[1, 1]
monetary = rfm.groupby('segment_name')['monetary_180d'].mean().reindex(counts.index)
ax4.barh(monetary.index, monetary.values, color=plt.cm.viridis(np.linspace(0, 1, len(monetary))))
ax4.set_xlabel('Average Spend (180d)')
ax4.set_title('Customer Value by Segment')
ax4.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('rfm_analysis_charts.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Final Segments

In [ ]:
# Save segments.csv with all required features
output_cols = ['customer_id', 'segment_name', 'recency_days', 'frequency_180d', 'monetary_180d',
               'R_score', 'F_score', 'M_score', 'RFM_score',
               'return_rate_180d', 'avg_discount_pct_180d', 'avg_rating_180d',
               'ticket_count_90d', 'negative_ticket_rate_90d',
               'sessions_30d', 'campaign_clicks_30d', 'email_opens_30d',
               'churn_next_60d', 'split',
               'support_complaint_score', 'engagement_score', 'discount_dependency',
               'loyalty_tier']

segments_df = rfm[output_cols].copy()
segments_df.to_csv('segments.csv', index=False)
print(f"✓ Saved segments.csv: {len(segments_df)} customers, {len(output_cols)} features")
print(f"✓ Segments: {segments_df['segment_name'].nunique()}")
print(f"\nFirst few rows:")
segments_df.head()

## 8. Manual Review Cases

Customers with mixed signals or borderline scores requiring human judgment.

In [ ]:
# Identify manual review cases
rfm['mixed_signals'] = (
    ((rfm['R_score'] >= 4) & (rfm['F_score'] <= 2)) |  # Recent but low freq
    ((rfm['M_score'] >= 4) & (rfm['R_score'] <= 2)) |  # High value but at risk
    ((rfm['engagement_score'] > 0.5) & (rfm['R_score'] <= 2)) |  # Engaged but not buying
    ((rfm['RFM_score'] >= 8) & (rfm['RFM_score'] <= 10))  # Borderline RFM
)

manual_review = rfm[rfm['mixed_signals']].head(10)
print(f"Manual review cases: {len(manual_review)}")
manual_review[['customer_id', 'segment_name', 'RFM_score', 'recency_days', 
               'frequency_180d', 'monetary_180d', 'churn_next_60d', 'engagement_score']]

## Summary

**Deliverables:**
- ✓ RFM features created (R, F, M scores 1-5)
- ✓ 9 customer segments with business logic
- ✓ 3 additional signals: support complaints, engagement, discount dependency
- ✓ segments.csv saved with all features
- ✓ 10 manual review cases identified

**Next Steps:**
- Review retention_strategy.md for segment-specific actions
- Check manual_review_cases.md for detailed case analysis
- Use segments.csv for targeted retention campaigns